# **EDA: El impacto de la IA**
## Limpieza y revisión de la calidad de los datos

Este notebook prepara los dos datasets para comprobar las tres hipótesis del proyecto:

1. Quien usa más IA es más productivo.
2. El uso excesivo de la IA tiene efectos negativos en el bienestar.
3. Los estudiantes que usan más IA sacan mejores notas, pero retienen menos lo aprendido.

Los pasos son: 
- Unir los dos ficheros del dataset de empleados. 
- Revisar la calidad de los datos (nulos, duplicados, rangos, valores en los límites y categorías). 
- Anotar las decisiones de limpieza que se toman.

Los tres datasets son sintéticos. Es decir, generados artificialmente. Por eso, además de buscar errores, se revisa cómo se han generado los datos.

# Dataset Empleados

## Paso 1. Carga y unión de los dos ficheros

El dataset de empleados viene en dos ficheros: `features` (perfil y hábitos de cada empleado) y `targets` (productividad y nivel de burnout). Se unen por `Employee_ID`.

In [1]:
import pandas as pd

df_features = pd.read_csv("../data/ai_productivity_features.csv")
df_targets = pd.read_csv("../data/ai_productivity_targets.csv")

df_empleados = pd.merge(df_features, df_targets, on="Employee_ID")
df_empleados.head()

,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,burnout_risk_score,productivity_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,10.00,81.0,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,6.78,59.2,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,9.66,62.4,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,10.00,76.8,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,5.38,53.7,Medium


Antes de seguir, se mira qué contiene cada fichero: columnas, tipos y valores no nulos.

In [2]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Employee_ID                   4500 non-null   str    
 1   job_role                      4500 non-null   str    
 2   experience_years              4500 non-null   int64  
 3   ai_tool_usage_hours_per_week  4500 non-null   float64
 4   tasks_automated_percent       4500 non-null   float64
 5   manual_work_hours_per_week    4500 non-null   float64
 6   learning_time_hours_per_week  4500 non-null   float64
 7   deadline_pressure_level       4500 non-null   str    
 8   meeting_hours_per_week        4500 non-null   float64
 9   collaboration_hours_per_week  4500 non-null   float64
 10  error_rate_percent            4500 non-null   float64
 11  task_complexity_score         4500 non-null   int64  
 12  focus_hours_per_day           4500 non-null   float64
 13  work_life_bala

In [3]:
df_targets.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Employee_ID         4500 non-null   str    
 1   productivity_score  4500 non-null   float64
 2   burnout_risk_level  4500 non-null   str    
dtypes: float64(1), str(2)
memory usage: 105.6 KB


Se comprueba que la unión es correcta: 4.500 filas, sin nulos ni duplicados y 17 columnas (las 15 de `features` más las 2 que aporta `targets`).

In [4]:
print(df_features.shape, df_targets.shape, df_empleados.shape)
print(df_empleados.isna().sum().sum())
print(df_empleados.duplicated().sum())
df_empleados.info()

(4500, 15) (4500, 3) (4500, 17)
0
0
<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Employee_ID                   4500 non-null   str    
 1   job_role                      4500 non-null   str    
 2   experience_years              4500 non-null   int64  
 3   ai_tool_usage_hours_per_week  4500 non-null   float64
 4   tasks_automated_percent       4500 non-null   float64
 5   manual_work_hours_per_week    4500 non-null   float64
 6   learning_time_hours_per_week  4500 non-null   float64
 7   deadline_pressure_level       4500 non-null   str    
 8   meeting_hours_per_week        4500 non-null   float64
 9   collaboration_hours_per_week  4500 non-null   float64
 10  error_rate_percent            4500 non-null   float64
 11  task_complexity_score         4500 non-null   int64  
 12  focus_hours_per_day           4500 no

`merge` añade las columnas de `targets` al final, pero `burnout_risk_score` venía en `features` y queda en medio. Se reordenan las columnas para que las tres de resultado (productividad y burnout) queden juntas al final de la tabla.

In [5]:
columns_orden = ["productivity_score", "burnout_risk_score", "burnout_risk_level"]
columns_resto = [c for c in df_empleados.columns if c not in columns_orden]

df_empleados_2 = df_empleados[columns_resto + columns_orden]
df_empleados_2.head()

,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,productivity_score,burnout_risk_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,81.0,10.00,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,59.2,6.78,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,62.4,9.66,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,76.8,10.00,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,53.7,5.38,Medium


## Paso 2. Revisión de la calidad de los datos

### Rangos de las variables numéricas

`describe()` resume cada variable (mínimo, máximo, media y cuartiles) y `.T` traspone la tabla para leerla mejor. Se buscan valores imposibles y valores que parezcan un tope.

In [6]:
df_empleados_2.describe().T

,count,mean,std,min,25%,50%,75%,max
experience_years,4500.0,10.427556,5.745073,1.00,6.000,10.000,15.00,20.00
ai_tool_usage_hours_per_week,4500.0,10.349667,5.808240,0.00,6.000,9.700,14.30,30.40
tasks_automated_percent,4500.0,25.908467,15.157930,0.00,14.500,24.500,36.00,79.90
manual_work_hours_per_week,4500.0,22.372867,6.465516,10.00,17.900,22.500,26.80,43.30
learning_time_hours_per_week,4500.0,2.001756,1.408176,0.00,1.000,1.700,2.70,11.50
meeting_hours_per_week,4500.0,6.571089,4.683117,1.00,3.400,5.700,8.20,30.20
collaboration_hours_per_week,4500.0,8.249311,3.040733,2.00,6.200,8.000,10.00,20.00
error_rate_percent,4500.0,2.097044,1.519145,0.10,0.710,2.000,3.29,6.67
task_complexity_score,4500.0,5.507556,2.861737,1.00,3.000,6.000,8.00,10.00
focus_hours_per_day,4500.0,4.694489,1.337197,1.00,3.900,4.800,5.60,8.00


**Qué se observa:**

- No hay valores imposibles (horas negativas o porcentajes por encima de 100).
- La columna `count` es 4.500 en todas las variables, no faltan datos.
- **`burnout_risk_score`:** El máximo es 10 y el percentil 75 también es 10, así que como mínimo el 25 % de los empleados tiene exactamente 10. La media es 8,35: casi todos están en la parte alta. Esto limita la hipótesis 2 porque hay poca variación para ver diferencias.
- **`productivity_score`:** El máximo es 100, pero se comprueba cuántos empleados lo alcanzan.
- **`error_rate_percent`:** El mínimo es 0,10, lo que parece un suelo. La ficha de Kaggle no define esta variable, se interpreta como el porcentaje de errores en el trabajo.
- **`manual_work_hours_per_week`, `collaboration_hours_per_week` y `focus_hours_per_day`:** Tienen mínimos y máximos redondos (10,0; 2,0 a 20,0 y 1,0 a 8,0), por lo que probablemente están acotados.
- **`experience_years`:** Va de 1 a 20 años y cada año tiene entre 185 y 254 empleados, un reparto casi uniforme. En una plantilla real habría muchos más empleados con poca experiencia.

### Valores en los límites

Cuando una variable tiene decimales, es poco habitual que mucha gente repita exactamente el mismo valor extremo. Si ocurre, probablemente los valores se calcularon por encima del límite y luego se cortaron. 

Se mide qué porcentaje de empleados está justo en cada límite.

In [7]:
print("Burnout = 10:", round((df_empleados_2["burnout_risk_score"] == 10).mean() * 100, 1), "%")
print("Productividad = 100:", round((df_empleados_2["productivity_score"] == 100).mean() * 100, 1), "%")
print("Errores = 0,10:", round((df_empleados_2["error_rate_percent"] == 0.10).mean() * 100, 1), "%")
print("Equilibrio vida-trabajo = 1:", round((df_empleados_2["work_life_balance_score"] == 1).mean() * 100, 1), "%")
print("Equilibrio vida-trabajo = 10:", round((df_empleados_2["work_life_balance_score"] == 10).mean() * 100, 1), "%")

Burnout = 10: 37.4 %
Productividad = 100: 0.7 %
Errores = 0,10: 14.5 %
Equilibrio vida-trabajo = 1: 4.3 %
Equilibrio vida-trabajo = 10: 0.2 %


| Variable | Resultado | Interpretación |
|---|---|---|
| Burnout (puntuación) = 10 | 37,4 % | Tope claro. Limita la hipótesis 2. |
| Errores = 0,10 | 14,5 % | Suelo. Parece un mínimo artificial. |
| Productividad = 100 | 0,7 % | Sin problema. |
| Equilibrio vida-trabajo = 1 | 4,3 % | Suelo leve. |
| Equilibrio vida-trabajo = 10 | 0,2 % | Sin problema. |

El burnout y los errores tienen mucha gente pegada al límite, mientras que la productividad y el equilibrio vida-trabajo casi no. Por eso el equilibrio vida-trabajo, que no está cortado, sirve como segunda medida de bienestar para la hipótesis 2.

### Categorías

Se cuenta cuántos empleados hay en cada categoría para ver si los grupos están equilibrados, porque comparar grupos pequeños da resultados poco fiables.

In [8]:
for column in ["job_role", "deadline_pressure_level", "burnout_risk_level"]:
    print(df_empleados_2[column].value_counts(), "\n")

job_role
Developer    1115
Analyst       892
Designer      722
Marketer      658
Manager       652
Writer        461
Name: count, dtype: int64 

deadline_pressure_level
Low       1527
High      1503
Medium    1470
Name: count, dtype: int64 

burnout_risk_level
High      3303
Medium    1087
Low        110
Name: count, dtype: int64 



- **`job_role`:** Los grupos son desiguales (de 1.115 developers a 461 writers), pero todos son lo bastante grandes para comparar.
- **`deadline_pressure_level`:** Casi un tercio en cada nivel (34 %, 33 % y 33 %). Un reparto tan igualado es poco habitual en datos reales y es otra señal de que son sintéticos.
- **`burnout_risk_level`:** 73 % High (3.303), 24 % Medium (1.087) y 2,4 % Low (110). Está muy desequilibrado: comparar el grupo Low con los demás es poco fiable.

### ¿Son `burnout_risk_score` y `burnout_risk_level` la misma información?

Se calculan las estadísticas de la puntuación dentro de cada nivel.

In [9]:
df_empleados_2.groupby("burnout_risk_level")["burnout_risk_score"].describe()

,count,mean,std,min,25%,50%,75%,max
burnout_risk_level,,,,,,,,
High,3303.0,9.301075,0.925743,7.01,8.6300,10.000,10.00,10.00
Low,110.0,4.050818,0.341860,3.22,3.8125,4.105,4.33,4.49
Medium,1087.0,5.915023,0.673401,4.54,5.3800,5.960,6.48,7.00


Los rangos no se solapan: Low va de 3,22 a 4,49, Medium de 4,54 a 7,00 y High de 7,01 a 10,00. Por tanto, el nivel es la puntuación dividida en tres tramos: es la misma información y no cuentan como dos pruebas distintas.

### Conclusiones y decisiones de limpieza (empleados)

- No hay nulos ni duplicados, así que no se elimina ninguna fila.
- Los dos ficheros se unen por `Employee_ID` (4.500 filas) y se reordenan las columnas para que las de productividad y burnout queden juntas al final de la tabla.
- Los valores en los límites (burnout, errores y equilibrio vida-trabajo) no se consideran errores: parecen límites de cómo se generó el dato. Si se eliminaran, se sesgaría el resultado. Así que se mantienen y se documentan con su porcentaje.
- Se usa `burnout_risk_score` como medida principal porque es más detallada. `burnout_risk_level` es esa misma puntuación dividida en tres tramos y solo se usa para presentar grupos.

- **Límites para el análisis:**
    - Con el 73 % de empleados en burnout High y el 37 % en el máximo hay poco margen para ver diferencias en la hipótesis 2. El equilibrio vida-trabajo ayuda a compensarlo.
    - El grupo Low tiene solo 110 personas, por lo que comparar con él es frágil.
    - La productividad está bien repartida, así que la hipótesis 1 no tiene este problema.

# Dataset Estudiantes

## Paso 1. Carga y comprobación

In [10]:
df_estudiantes = pd.read_csv("../data/ai_student_impact_dataset.csv")
df_estudiantes.head()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


Se comprueba el tamaño, los nulos y los duplicados y los tipos de cada columna.

In [11]:
print(df_estudiantes.shape)
print(df_estudiantes.isna().sum().sum())
print(df_estudiantes.duplicated().sum())
df_estudiantes.info()

(50000, 16)
0
0
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  50000 non-null  int64  
 1   Major_Category              50000 non-null  str    
 2   Year_of_Study               50000 non-null  str    
 3   Pre_Semester_GPA            50000 non-null  float64
 4   Weekly_GenAI_Hours          50000 non-null  float64
 5   Primary_Use_Case            50000 non-null  str    
 6   Prompt_Engineering_Skill    50000 non-null  str    
 7   Tool_Diversity              50000 non-null  int64  
 8   Paid_Subscription           50000 non-null  bool   
 9   Traditional_Study_Hours     50000 non-null  float64
 10  Perceived_AI_Dependency     50000 non-null  int64  
 11  Institutional_Policy        50000 non-null  str    
 12  Anxiety_Level_During_Exams  50000 non-null  int64  
 13  Post_Semester_GPA         

El dataset tiene 50.000 filas y 16 columnas, sin nulos ni duplicados.

## Paso 2. Revisión de la calidad de los datos

### Rangos de las variables numéricas

In [12]:
df_estudiantes.describe().T

,count,mean,std,min,25%,50%,75%,max
Student_ID,50000.0,125000.500000,14433.901067,100001.000,112500.75000,125000.500,137500.250,150000.000
Pre_Semester_GPA,50000.0,3.146102,0.478854,1.183,2.83400,3.210,3.521,3.998
Weekly_GenAI_Hours,50000.0,8.427752,8.269490,0.000,2.39000,5.800,11.720,40.000
Tool_Diversity,50000.0,2.800260,1.188020,1.000,2.00000,3.000,4.000,5.000
Traditional_Study_Hours,50000.0,11.209271,5.156426,1.000,7.56000,11.180,14.710,35.860
Perceived_AI_Dependency,50000.0,3.505360,1.820812,1.000,2.00000,3.000,5.000,10.000
Anxiety_Level_During_Exams,50000.0,4.270760,2.144066,1.000,3.00000,4.000,6.000,10.000
Post_Semester_GPA,50000.0,3.349299,0.495673,1.000,3.02375,3.421,3.749,4.000
Skill_Retention_Score,50000.0,75.798125,13.281626,10.780,66.82000,76.000,85.190,100.000


**Qué se observa:**

- **`Student_ID`:** Va de 100001 a 150000 en orden, con media 125.000,5. Es un identificador consecutivo y no una variable, así que no se incluye en el análisis.
- **`Pre_Semester_GPA` y `Post_Semester_GPA`:** La nota inicial va de 1,18 a 3,998 y la final de 1,0 a 4,0. La nota final está cortada en 4,0. La media sube de 3,15 a 3,35. Es decir, unos 0,2 puntos de mejora general.
- **`Weekly_GenAI_Hours`:** Va de 0 a 40 horas. La media (8,4) es mayor que la mediana (5,8) y la desviación (8,3) es casi igual a la media. La distribución es asimétrica, con mucha gente que usa poco y unos pocos que usan mucho. Esto importa al hacer los tramos.
- **`Skill_Retention_Score`:** Llega a 100 exactos, probablemente con un tope.
- **`Traditional_Study_Hours`** (de 1 a 35,86) y **`Pre_Semester_GPA`** no parecen cortadas en un valor redondo.

### Valores en los límites

Se mide qué porcentaje de estudiantes está justo en cada límite, y si cada `Student_ID` es único.

In [13]:
print("Nota final = 4,0:", round((df_estudiantes["Post_Semester_GPA"] == 4).mean() * 100, 2), "%")
print("Nota final = 1,0:", round((df_estudiantes["Post_Semester_GPA"] == 1).mean() * 100, 3), "%")
print("Horas de IA = 0:", round((df_estudiantes["Weekly_GenAI_Hours"] == 0).mean() * 100, 2), "%")
print("Horas de IA = 40:", round((df_estudiantes["Weekly_GenAI_Hours"] == 40).mean() * 100, 2), "%")
print("Retención = 100:", round((df_estudiantes["Skill_Retention_Score"] == 100).mean() * 100, 2), "%")
print("ID únicos:", df_estudiantes["Student_ID"].nunique())

Nota final = 4,0: 9.61 %
Nota final = 1,0: 0.002 %
Horas de IA = 0: 0.06 %
Horas de IA = 40: 0.9 %
Retención = 100: 4.29 %
ID únicos: 50000


| Variable | Resultado | Interpretación |
|---|---|---|
| Nota final = 4,0 | 9,6 % | Casi 1 de cada 10 no puede subir más. Afecta a la hipótesis 3. |
| Nota final = 1,0 | 0,002 % | Solo 1 alumno. Sin problema. |
| Horas de IA = 0 | 0,06 % (unos 28 alumnos) | Casi todos usan IA, así que no se puede comparar "usa frente a no usa", sino uso bajo frente a alto. |
| Horas de IA = 40 | 0,9 % | Tope leve. |
| Retención = 100 | 4,3 % | Tope leve. |
| ID únicos | 50.000 | Sin alumnos repetidos. |

### Categorías

Se cuenta cuántos estudiantes hay en cada categoría para ver si los grupos están equilibrados.

In [14]:
for column in ["Major_Category", "Year_of_Study", "Primary_Use_Case",
               "Prompt_Engineering_Skill", "Institutional_Policy",
               "Burnout_Risk_Level", "Paid_Subscription"]:
    print(df_estudiantes[column].value_counts(), "\n")

Major_Category
STEM          15059
Business      12538
Humanities     9994
Medical        6476
Arts           5933
Name: count, dtype: int64 

Year_of_Study
Junior       11045
Freshman     11031
Senior       10634
Sophomore     9860
Graduate      7430
Name: count, dtype: int64 

Primary_Use_Case
Debugging/Troubleshooting    12295
Copywriting/Drafting         12011
Ideation                     10721
Summarizing_Reading           8633
Direct_Answer_Generation      6340
Name: count, dtype: int64 

Prompt_Engineering_Skill
Beginner        18495
Intermediate    17696
Advanced        13809
Name: count, dtype: int64 

Institutional_Policy
Allowed_With_Citation    25224
Actively_Encouraged      14988
Strict_Ban                9788
Name: count, dtype: int64 

Burnout_Risk_Level
Medium    21144
Low       16369
High      12487
Name: count, dtype: int64 

Paid_Subscription
False    28846
True     21154
Name: count, dtype: int64 



- Todos los grupos son grandes (el más pequeño, `Direct_Answer_Generation`, tiene 6.340 estudiantes, un 12,7 %), así que se pueden comparar. No están repartidos por igual.
- **`Major_Category`:** de STEM (30 %) a Arts (12 %).
- **`Prompt_Engineering_Skill`:** 37 % Beginner, 35 % Intermediate y 28 % Advanced.
- **`Institutional_Policy`:** 50 % permitido con cita, 30 % fomentado y 20 % prohibido.
- **`Burnout_Risk_Level`:** Está bien repartido (42 % Medium, 33 % Low y 25 % High). Al contrario que en empleados, aquí hay suficientes estudiantes en cada nivel para comparar.

### Conclusiones y decisiones de limpieza (estudiantes)

- No hay nulos ni duplicados, así que no se elimina ninguna fila.
- Los valores en los límites (nota final, horas de IA y retención) se mantienen y se documentan con su porcentaje.
- Se descarta `Student_ID` del análisis porque es solo un identificador.
- **Límites para el análisis:**
    - El 9,6 % de los estudiantes termina con un 4,0, así que no puede mejorar más. Se tendrá en cuenta al analizar la mejora de nota en la hipótesis 3.
    - Casi todos los estudiantes usan IA. Se compara uso bajo frente a uso alto, no usuarios frente a no usuarios.